# Prerequisites
본 `ipynb` 은 `Python=3.12` 에서 작성하였습니다. Package dependency 를 해결하기 위해 아래 cell 을 실행해주세요.

## Install Python packages

In [ ]:
%pip -q install -U openai==2.36.0 sounddevice==0.5.5 dotenv==0.9.9 websocket-client==1.9.0 azure-search-documents==11.7.0b2 "azure-ai-voicelive[aiohttp]==1.2.0"

## Load environment variables from a .env file
secret 노출을 피하고 notebook 들간의 일관된 환경변수를 설정하기 위해 `dotenv` 을 이용한다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_OPENAI_URI = os.getenv("AZURE_OPENAI_URI")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_REALTIME_DEPLOYMENT = os.getenv("AZURE_OPENAI_REALTIME_DEPLOYMENT")
AZURE_AI_SEARCH_ENDPOINT = os.getenv("AZURE_AI_SEARCH_ENDPOINT")
AZURE_AI_SEARCH_ADMIN_KEY = os.getenv("AZURE_AI_SEARCH_ADMIN_KEY")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_AI_SEARCH_INDEX = "news-ko-index"

# RAG pipeline using voice models

## Upload documents into AI Search
RAG 에 사용될 documents 들을 AI Search 에 업로드한다.

In [ ]:
# resources/news_ko.csv 를 Azure AI Search 인덱스(news-ko-index)에 업로드한다.
# 인덱스 스키마는 JSON 이 아니라 azure-search-documents SDK 모델(코드)로 정의한다.
import csv

from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import ResourceNotFoundError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchProfile,
    VectorSearchAlgorithmMetric,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)
from openai import OpenAI

EMBED_DIM = 3072        # text-embedding-3-large 출력 차원
EMBED_MAX_CHARS = 6000  # 임베딩 토큰 한도(8191) 초과 방지용 글자수 제한
CSV_PATH = "resources/news_ko.csv"

cred = AzureKeyCredential(AZURE_AI_SEARCH_ADMIN_KEY)

# --- 1) 인덱스 스키마를 코드로 정의 (id, summary, category, content, embedding3) ---
index = SearchIndex(
    name=AZURE_AI_SEARCH_INDEX,
    fields=[
        # 키 필드
        SimpleField(
            name="id", type=SearchFieldDataType.String,
            key=True, filterable=True, sortable=True,
        ),
        # 한국어 분석기로 검색 가능한 요약 (큰 텍스트라 facet 비활성)
        SearchableField(
            name="summary", type=SearchFieldDataType.String,
            analyzer_name="ko.lucene", facetable=False,
        ),
        # 분류 (필터/패싯용)
        SimpleField(
            name="category", type=SearchFieldDataType.String,
            filterable=True, facetable=True, sortable=True,
        ),
        # 한국어 분석기로 검색 가능한 본문 (큰 텍스트라 facet 비활성)
        SearchableField(
            name="content", type=SearchFieldDataType.String,
            analyzer_name="ko.lucene", facetable=False,
        ),
        # 벡터 필드 (조회/저장 안 함, HNSW 프로파일 연결)
        SearchField(
            name="embedding3",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            hidden=True,    # retrievable=False
            stored=False,   # 원본 벡터 미저장으로 용량 절약
            vector_search_dimensions=EMBED_DIM,
            vector_search_profile_name="embedding3-profile",
        ),
    ],
    # 벡터 검색 설정 (HNSW + 코사인)
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="hnsw_config",
                parameters=HnswParameters(
                    metric=VectorSearchAlgorithmMetric.COSINE,
                    m=4, ef_construction=400, ef_search=500,
                ),
            ),
        ],
        profiles=[
            VectorSearchProfile(
                name="embedding3-profile",
                algorithm_configuration_name="hnsw_config",
            ),
        ],
    ),
    # 시맨틱 랭킹 설정
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="summary"),
                    content_fields=[SemanticField(field_name="content")],
                    keywords_fields=[SemanticField(field_name="category")],
                ),
            ),
        ],
    ),
)

# 코드의 스키마가 기준이 되도록 기존 인덱스가 있으면 지우고 새로 만든다
index_client = SearchIndexClient(endpoint=AZURE_AI_SEARCH_ENDPOINT, credential=cred)
try:
    index_client.delete_index(AZURE_AI_SEARCH_INDEX)
    print(f"🗑️  기존 인덱스 '{AZURE_AI_SEARCH_INDEX}' 삭제")
except ResourceNotFoundError:
    pass
index_client.create_index(index)
print(f"✅ 인덱스 '{AZURE_AI_SEARCH_INDEX}' 생성 완료")

# --- 2) CSV 로드 후 content 를 임베딩 (azure_ai_search 와 동일한 임베딩 모델 사용) ---
with open(CSV_PATH, encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))
print(f"📄 CSV {len(rows)}건 로드: {CSV_PATH}")

embed_client_sync = OpenAI(
    base_url=AZURE_OPENAI_URI.rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)
texts = [(r.get("content") or " ")[:EMBED_MAX_CHARS] for r in rows]
emb_resp = embed_client_sync.embeddings.create(
    model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=texts,
)
vectors = [d.embedding for d in emb_resp.data]
print(f"🧠 임베딩 {len(vectors)}건 완료 (차원 {len(vectors[0])})")

# --- 3) 문서로 변환 후 업로드 ---
documents = [
    {
        "id": r["id"],
        "summary": r.get("summary", ""),
        "category": r.get("category", ""),
        "content": r.get("content", ""),
        "embedding3": vec,
    }
    for r, vec in zip(rows, vectors)
]
search_client = SearchClient(
    endpoint=AZURE_AI_SEARCH_ENDPOINT,
    index_name=AZURE_AI_SEARCH_INDEX,
    credential=cred,
)
result = search_client.upload_documents(documents=documents)
ok = sum(1 for x in result if x.succeeded)
print(f"⬆️  업로드 완료: {ok}/{len(documents)} 성공")


## gpt-realtime
voice-basic.ipynb 에서 했던 gpt-realtime 예제에 RAG 를 위한 tool calling 을 추가해보자.

### 동작 원리 (블록도)

```text
   🎙️ 마이크 (PCM 24kHz)
        │  ▶ input_audio_buffer.append
        │     (반이중: AI 발화 중엔 마이크 전송 보류 → 끊김 방지)
        ▼
  ┌─ gpt-realtime (서버) ─────────────────────────────────
  │   ① Server VAD       : 발화 시작/종료 감지 → 턴 분리
  │   ② Speech-to-Speech : 음성을 직접 이해·추론·생성
  │   ③ 도구 필요 판단    : search 함수 호출을 요청
  └───┬───────────────────────────────┬────────────────────
      │ ⓐ 일반 음성 응답               │ ⓑ 도구 호출 (RAG)
      │                               ▼  response.function_call_arguments.done
      │                          ┌─ azure_ai_search(query) ──────────
      │                          │  1) 쿼리 임베딩 (text-embedding-3-large)
      │                          │  2) 하이브리드(키워드+벡터)+시맨틱
      │                          │     → Azure AI Search (news-ko-index)
      │                          │  3) 검색 결과 텍스트 반환
      │                          └───┬───────────────────────────────
      │                              │ function_call_output → response.create
      │                              ▼  (검색 결과를 근거로 음성 답변 재생성)
      ▼  ◀ response.output_audio.delta
   🔊 스피커 (PCM 24kHz)

   ↕ 전 구간이 하나의 WebSocket 양방향(full-duplex) 스트림 + 반이중 제어
```

기본 gpt-realtime(음성↔음성) 위에 **function calling 으로 RAG** 를 얹은 구조다.
모델이 사실·지식 질문이라고 판단하면 `search` 도구를 호출 → `azure_ai_search` 가
Azure AI Search(`news-ko-index`)를 검색 → 그 결과를 근거로 음성 답변을 생성한다.

**아래 코드의 이벤트가 위 흐름에 그대로 대응한다.**

| 단계 | Realtime 이벤트 |
|------|-----------------|
| 마이크 → 서버 전송 (반이중) | `input_audio_buffer.append` |
| 사용자 발화 감지 | `input_audio_buffer.speech_started` |
| 도구 호출 인자 완성 | `response.function_call_arguments.done` |
| 검색 결과 회신 | `conversation.item.create` (function_call_output) → `response.create` |
| 응답 오디오 수신(재생) | `response.output_audio.delta` |
| 응답 종료 | `response.done` |


In [ ]:
import json
import asyncio
import base64
import threading

import httpx
import sounddevice as sd
from openai import AsyncOpenAI

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 간단한 스피커 래퍼 클래스 (지터 버퍼로 끊김 완화)
class Speaker:
    def __init__(self, prebuffer_ms=150):
        self.buf = bytearray()
        self.lock = threading.Lock()
        self.playing = False
        # 선버퍼링 임계치(바이트): 24kHz, 모노, 16bit(=2byte) 기준
        self.prebuffer = int(24000 * 2 * prebuffer_ms / 1000)
        # latency="high" 로 PortAudio 내부 버퍼를 키워 언더런(끊김) 여유 확보
        self.stream = sd.RawOutputStream(
            samplerate=24000, channels=1, dtype="int16",
            blocksize=0, latency="high", callback=self._cb,
        )

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        # PortAudio 콜백은 별도의 스레드에서 호출되므로, 버퍼 접근을 락으로 보호
        with self.lock:
            # 선버퍼링: 충분히 쌓이기 전엔 무음을 내보내 네트워크 지터를 흡수
            if not self.playing:
                if len(self.buf) < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True
            avail = min(need, len(self.buf))
            outdata[:avail] = bytes(self.buf[:avail])
            del self.buf[:avail]
            if avail < need:
                outdata[avail:] = b"\x00" * (need - avail)
                # 언더런 발생 → 다시 선버퍼링 모드로 전환해 끊김 누적 방지
                self.playing = False

    def add(self, pcm: bytes):
        # PCM 데이터를 버퍼에 추가하는 메서드. 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        # 버퍼를 초기화하는 메서드. 콜백과 동시에 호출될 수 있으므로 락으로 보호
        with self.lock:
            self.buf.clear()
            self.playing = False  # 다음 응답은 다시 선버퍼링부터

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


# PortAudio 초기화 및 재설정
reset_portaudio()

# OpenAI 클라이언트 초기화 (gpt-realtime 은 WebSocket 연결)
client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.replace("https://", "wss://").rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

# 임베딩 전용 클라이언트 (realtime 은 wss, 임베딩 호출은 https 라 분리해서 생성)
embed_client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

speaker, mic, sender = None, None, None
# 반이중(half-duplex) 상태: AI가 말하는 동안엔 마이크 입력을 서버로 보내지 않아
# 잡음/숨소리로 인한 오인식(VAD)이 AI 발화를 끊지 못하게 한다.
dialog_state = {"assistant_speaking": False}
try:
    # 스피커와 마이크 초기화
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    # 마이크 콜백 함수: PCM 데이터를 받아서 asyncio 큐에 넣음
    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    # 마이크 스트림 설정: 24000Hz, 모노, 16비트 PCM
    # blocksize=2400(=100ms) 로 잡아 전송 빈도를 낮춰(초당 ~10회) 이벤트 루프 부담을 줄임
    # → 들어오는 재생 오디오 처리가 밀리지 않아 끊김 완화
    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16",
        blocksize=2400, latency="high", callback=mic_cb,
    )

    # OpenAI 실시간 연결 설정 및 이벤트 루프
    async with client.realtime.connect(model=AZURE_OPENAI_REALTIME_DEPLOYMENT) as conn:
        await conn.session.update(session={
            "type": "realtime",
            # 도구를 적극 사용하도록 지시 (안 그러면 모델이 search 를 호출할 이유가 없음)
            "instructions": (
                "너는 친절한 한국어 AI 비서야. 사용자가 사실·지식·뉴스에 관해 물으면 "
                "추측하지 말고 반드시 search 도구를 호출해 검색한 뒤, 그 결과를 근거로 "
                "한국어로 간결하게 음성으로 답해줘. 검색 결과에 없는 내용이면 모른다고 솔직히 말해."
            ),
            "output_modalities": ["audio"],
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": 24000},
                    "transcription": {"model": "gpt-4o-transcribe", "language": "ko"}, # 실시간 음성 인식
                    "turn_detection": {
                        "type": "server_vad",   # 서버에서 음성 활동 감지(VAD) 사용
                        "threshold": 0.6,   # VAD 민감도(↑ 둔감) — 잡음 오인식 줄이려 0.5→0.6
                        "prefix_padding_ms": 300,   # 음성 시작 전 패딩
                        "silence_duration_ms": 700, # 무음 지속 판단(↑) — 짧은 끊김 오인식 방지
                        "create_response": True,
                        "interrupt_response": True, # (반이중이라 AI 발화 중엔 마이크를 안 보내 사실상 끼어들기 없음)
                    },
                },
                "output": {
                    "voice": "marin",
                    "format": {"type": "audio/pcm", "rate": 24000},
                },
            },
            # 실시간 대화에서 도구 사용 예시
            "tools": [{
                "type": "function",
                "name": "search",
                "description": (
                    "한국어 지식/뉴스 기사(인공지능·전기차·비트코인·손흥민·BTS·김치·제주도 등)가 "
                    "색인된 Azure AI Search 에서 관련 정보를 검색합니다."
                ),
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "검색어",
                        },
                    },
                    "required": ["query"],
                },
            }],
            "tool_choice": "auto",
        })
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. 예를 들어 '손흥민에 대해 알려줘', '인공지능 역사에 대해 알려줘' 라고 말해보세요. (Ctrl+C 로 종료)\n")

        # 마이크 → OpenAI 전송 태스크. AI가 말하는 동안(assistant_speaking)엔
        # 큐를 비우기만 하고 서버로는 보내지 않아(반이중) AI 발화가 끊기지 않게 한다.
        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                if dialog_state["assistant_speaking"]:
                    continue
                await conn.input_audio_buffer.append(
                    audio=base64.b64encode(chunk).decode(),
                )

        # 도구 호출 시 Azure AI Search(news-ko-index)를 실제로 조회하는 함수
        # - 쿼리를 임베딩한 뒤 하이브리드(키워드 summary,content + 벡터 embedding3) + 시맨틱 랭킹으로 검색
        # - 반환값은 모델이 음성 답변에 쓸 수 있도록 정리한 문자열 (function_call_output.output 은 문자열)
        # - httpx/AsyncOpenAI 모두 async 라 음성 이벤트 루프를 막지 않음
        async def azure_ai_search(query: str, k: int = 3) -> str:
            # 1) 쿼리 임베딩
            emb = await embed_client.embeddings.create(
                model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=[query],
            )
            qvec = emb.data[0].embedding

            # 2) 하이브리드 + 시맨틱 검색 요청
            body = {
                "search": query,
                "searchFields": "summary,content",
                "vectorQueries": [
                    {"kind": "vector", "vector": qvec, "fields": "embedding3", "k": k},
                ],
                "queryType": "semantic",
                "semanticConfiguration": "default",
                "select": "category,summary,content",
                "top": k,
            }
            url = (
                f"{AZURE_AI_SEARCH_ENDPOINT}/indexes/{AZURE_AI_SEARCH_INDEX}"
                f"/docs/search?api-version=2024-07-01"
            )
            async with httpx.AsyncClient(timeout=30) as http:
                resp = await http.post(url, json=body, headers={
                    "api-key": AZURE_AI_SEARCH_ADMIN_KEY,
                    "Content-Type": "application/json",
                })
                resp.raise_for_status()
                results = resp.json().get("value", [])

            if not results:
                return "관련 정보를 찾지 못했습니다."

            # 3) 모델이 활용할 컨텍스트로 정리 (본문은 길이 제한, 위키 본문이 요약으로 시작하므로 content 만 사용)
            chunks = []
            for r in results:
                text = (r.get("content") or "").strip().replace("\n", " ")[:500]
                chunks.append(f"[{r.get('category')}] {text}")
            return "\n\n".join(chunks)

        sender = asyncio.create_task(pump_mic())
        # OpenAI 이벤트 처리 루프
        async for event in conn:
            # 이벤트 타입에 따라 스피커에 PCM 데이터를 추가하거나, 인식된 텍스트를 출력하거나, 에러를 처리
            if event.type == "response.output_audio.delta":
                speaker.add(base64.b64decode(event.delta))
            # AI 응답 시작: 반이중 모드로 마이크 입력을 잠시 막아 AI 발화가 끊기지 않게 함
            elif event.type == "response.created":
                dialog_state["assistant_speaking"] = True
            # 서버 VAD가 음성 활동을 감지하여 turn이 시작되었을 때, 이전 대화 내용을 초기화
            elif event.type == "response.output_audio_transcript.delta":
                print(event.delta, end="", flush=True)
            # 서버 VAD가 음성 활동이 종료되었다고 판단하여 turn이 완료되었을 때, 최종 대화 내용을 출력하고 스피커 버퍼 초기화
            elif event.type == "conversation.item.input_audio_transcription.completed":
                print(f"\n🧑 {event.transcript}")
            # 서버 VAD가 음성 활동이 시작되었다고 판단하여 turn이 시작되었을 때, 스피커 버퍼 초기화
            elif event.type == "input_audio_buffer.speech_started":
                print("\n🎧 음성 인식 시작...")
                speaker.clear()
            # AI 응답 완료: 마이크 입력 재개
            elif event.type == "response.done":
                dialog_state["assistant_speaking"] = False
                print("\n✅ 응답 완료")
                print()
            # 도구 호출: 인자가 완성되면 Azure AI Search 를 호출하고 결과를 모델에 되돌려준 뒤 응답 재생성
            elif event.type == "response.function_call_arguments.done":
                args = json.loads(event.arguments)
                print(f"\n🔧 도구 호출: {event.name}({args})")
                docs = await azure_ai_search(args["query"])
                print(f"🔍 검색 결과: {docs[:200]}...")
                await conn.conversation.item.create(item={
                    "type": "function_call_output",
                    "call_id": event.call_id,
                    "output": docs,
                })
                await conn.response.create()
            elif event.type == "error":
                dialog_state["assistant_speaking"] = False
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")

## Voice live

**Voice Live API 도 tool calling(function calling) 을 지원**한다. 그래서 위 gpt-realtime 셀에서
직접 WebSocket 으로 구현한 RAG 를, `azure-ai-voicelive` SDK 로 동일하게 만들 수 있다.
세션에 `FunctionTool` 로 `search` 도구를 등록하면, 모델이 사실·지식 질문에서 도구를 호출하고
`azure_ai_search` 가 Azure AI Search(`news-ko-index`)를 검색해 그 결과로 음성 답변을 생성한다.
또한 에코 제거·노이즈 억제가 서버에서 처리되므로 반이중 같은 수동 처리가 필요 없다.

### 동작 원리 (블록도) — Voice Live + tool calling

```text
   🎙️ 마이크 ─ input_audio_buffer.append ─▶
  ╔═ Voice Live API (WebSocket) ══════════════════════════════════
  ║   에코제거·노이즈억제 → Server VAD → gpt-realtime
  ║                                          │ 도구 필요 판단
  ╚══════════════════════════════════════════┼═══════════════════
        │ ⓐ 일반 음성 응답                    ▼ ⓑ RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE
        │                            ┌─ azure_ai_search(query) ──────────
        │                            │  쿼리 임베딩 → 하이브리드+시맨틱
        │                            │  → Azure AI Search (news-ko-index)
        │                            └───┬───────────────────────────────
        │                                │ conversation.item.create(FunctionCallOutputItem)
        │                                ▼ + response.create
        ◀ RESPONSE_AUDIO_DELTA ── 🔊 스피커   (검색 결과를 근거로 음성 답변)
```

**아래 코드의 이벤트 / API 대응**

| 단계 | Voice Live API |
|------|----------------|
| 도구 등록 | `RequestSession(tools=[FunctionTool(...)], tool_choice=ToolChoiceLiteral.AUTO)` |
| 마이크 → 서버 | `input_audio_buffer.append` |
| 발화 감지 | `ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED` |
| 도구 호출 인자 완성 | `ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE` |
| 검색 결과 회신 | `conversation.item.create(FunctionCallOutputItem(...))` → `response.create()` |
| 응답 오디오(재생) | `ServerEventType.RESPONSE_AUDIO_DELTA` (이미 디코딩된 bytes) |
| 응답 종료 | `ServerEventType.RESPONSE_DONE` |


In [ ]:
import json
import asyncio
import base64
import threading

import httpx
import sounddevice as sd
from openai import AsyncOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    RequestSession,
    Modality,
    InputAudioFormat,
    OutputAudioFormat,
    ServerVad,
    ServerEventType,
    AzureStandardVoice,
    AudioEchoCancellation,
    AudioNoiseReduction,
    FunctionTool,
    ToolChoiceLiteral,
    FunctionCallOutputItem,
)

# 재실행 시 남아 있는 PortAudio 상태 초기화 (-9986 예방)
def reset_portaudio():
    try:
        sd._terminate()
    except Exception as e:
        print(f"PortAudio reset error: {e}")
    sd._initialize()

# 지터 버퍼 스피커 (재생 끊김 완화)
class Speaker:
    def __init__(self, prebuffer_ms=150):
        self.buf = bytearray()
        self.lock = threading.Lock()
        self.playing = False
        self.prebuffer = int(24000 * 2 * prebuffer_ms / 1000)  # 24kHz·16bit 기준 바이트
        self.stream = sd.RawOutputStream(
            samplerate=24000, channels=1, dtype="int16",
            blocksize=0, latency="high", callback=self._cb,
        )

    def _cb(self, outdata, frames, time, status):
        need = len(outdata)
        with self.lock:
            if not self.playing:  # 선버퍼링
                if len(self.buf) < self.prebuffer:
                    outdata[:] = b"\x00" * need
                    return
                self.playing = True
            avail = min(need, len(self.buf))
            outdata[:avail] = bytes(self.buf[:avail])
            del self.buf[:avail]
            if avail < need:
                outdata[avail:] = b"\x00" * (need - avail)
                self.playing = False

    def add(self, pcm: bytes):
        with self.lock:
            self.buf.extend(pcm)

    def clear(self):
        with self.lock:
            self.buf.clear()
            self.playing = False

    def start(self):
        self.stream.start()

    def close(self):
        try:
            self.stream.stop()
        finally:
            self.stream.close()


# 임베딩 전용 클라이언트 (검색 쿼리 벡터화용, https v1 엔드포인트)
embed_client = AsyncOpenAI(
    base_url=AZURE_OPENAI_URI.rstrip("/") + "/openai/v1",
    api_key=AZURE_OPENAI_API_KEY,
)

# 도구 호출 시 Azure AI Search(news-ko-index)를 조회하는 함수 (gpt-realtime 셀과 동일)
async def azure_ai_search(query: str, k: int = 3) -> str:
    emb = await embed_client.embeddings.create(
        model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT, input=[query],
    )
    qvec = emb.data[0].embedding
    body = {
        "search": query,
        "searchFields": "summary,content",
        "vectorQueries": [
            {"kind": "vector", "vector": qvec, "fields": "embedding3", "k": k},
        ],
        "queryType": "semantic",
        "semanticConfiguration": "default",
        "select": "category,summary,content",
        "top": k,
    }
    url = (
        f"{AZURE_AI_SEARCH_ENDPOINT}/indexes/{AZURE_AI_SEARCH_INDEX}"
        f"/docs/search?api-version=2024-07-01"
    )
    async with httpx.AsyncClient(timeout=30) as http:
        resp = await http.post(url, json=body, headers={
            "api-key": AZURE_AI_SEARCH_ADMIN_KEY,
            "Content-Type": "application/json",
        })
        resp.raise_for_status()
        results = resp.json().get("value", [])
    if not results:
        return "관련 정보를 찾지 못했습니다."
    chunks = []
    for r in results:
        text = (r.get("content") or "").strip().replace("\n", " ")[:500]
        chunks.append(f"[{r.get('category')}] {text}")
    return "\n\n".join(chunks)

# Voice Live 에 등록할 search 도구 정의
search_tool = FunctionTool(
    name="search",
    description=(
        "한국어 지식/뉴스 기사(인공지능·전기차·비트코인·손흥민·BTS·김치·제주도 등)가 "
        "색인된 Azure AI Search 에서 관련 정보를 검색합니다."
    ),
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string", "description": "검색어"}},
        "required": ["query"],
    },
)

reset_portaudio()
speaker, mic, sender = None, None, None
try:
    speaker = Speaker()
    loop = asyncio.get_running_loop()
    mic_q: asyncio.Queue[bytes] = asyncio.Queue()

    def mic_cb(indata, frames, time, status):
        loop.call_soon_threadsafe(mic_q.put_nowait, bytes(indata))

    mic = sd.RawInputStream(
        samplerate=24000, channels=1, dtype="int16",
        blocksize=2400, latency="high", callback=mic_cb,
    )

    # Voice Live 연결 + 세션에 search 도구 등록
    async with connect(
        endpoint=AZURE_OPENAI_URI,
        credential=AzureKeyCredential(AZURE_OPENAI_API_KEY),
        model=AZURE_OPENAI_CHAT_DEPLOYMENT,
    ) as conn:
        await conn.session.update(session=RequestSession(
            modalities=[Modality.TEXT, Modality.AUDIO],
            instructions=(
                "너는 친절한 한국어 AI 비서야. 사용자가 사실·지식·뉴스에 관해 물으면 "
                "추측하지 말고 반드시 search 도구를 호출해 검색한 뒤, 그 결과를 근거로 "
                "한국어로 간결하게 음성으로 답해줘. 검색 결과에 없는 내용이면 모른다고 솔직히 말해."
            ),
            voice=AzureStandardVoice(name="ko-kr-SunHi:DragonHDLatestNeural"),
            input_audio_format=InputAudioFormat.PCM16,
            output_audio_format=OutputAudioFormat.PCM16,
            turn_detection=ServerVad(threshold=0.5, prefix_padding_ms=300, silence_duration_ms=500),
            input_audio_echo_cancellation=AudioEchoCancellation(),
            input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
            tools=[search_tool],
            tool_choice=ToolChoiceLiteral.AUTO,
        ))
        speaker.start()
        mic.start()
        print("🎙️  말해보세요. 예: '손흥민에 대해 알려줘', '비트코인이 뭐야'. (Ctrl+C 로 종료)\n")

        async def pump_mic():
            while True:
                chunk = await mic_q.get()
                await conn.input_audio_buffer.append(audio=base64.b64encode(chunk).decode())

        sender = asyncio.create_task(pump_mic())
        # Voice Live 이벤트 처리 루프
        async for event in conn:
            # Voice Live 의 오디오 delta 는 이미 디코딩된 bytes
            if event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                speaker.add(event.delta)
            elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA:
                print(event.delta, end="", flush=True)
            elif event.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                print(f"\n🧑 {event.transcript}")
            elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
                print("\n🎧 음성 인식 시작...")
                speaker.clear()  # 사용자가 말하기 시작하면 재생 중단(끼어들기)
            # 도구 호출: 인자가 완성되면 검색 후 결과를 모델에 되돌려주고 응답 재생성
            elif event.type == ServerEventType.RESPONSE_FUNCTION_CALL_ARGUMENTS_DONE:
                args = json.loads(event.arguments)
                print(f"\n🔧 도구 호출: {event.name}({args})")
                docs = await azure_ai_search(args["query"])
                print(f"🔍 검색 결과: {docs[:200]}...")
                await conn.conversation.item.create(
                    item=FunctionCallOutputItem(call_id=event.call_id, output=docs)
                )
                await conn.response.create()
            elif event.type == ServerEventType.RESPONSE_DONE:
                print("\n✅ 응답 완료\n")
            elif event.type == ServerEventType.ERROR:
                print("\n⚠️  ERROR:", event.error)
finally:
    if sender:
        sender.cancel()
    if mic:
        try:
            mic.stop(); mic.close()
        except Exception as e:
            print(f"마이크 종료 에러: {e}")
    if speaker:
        try:
            speaker.close()
        except Exception as e:
            print(f"스피커 종료 에러: {e}")
